<H3> Import Modules

In [99]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
print("Module Imported Successfully...")

Module Imported Successfully...


<h3> Labeled Smile Faces with 0 and Non-Smile with 1

In [100]:
SmileFolder = "smile"
NonSmileFolder = "non_smile"

data = []

if os.path.exists(SmileFolder):
    for f in os.listdir(SmileFolder):
        if f.lower().endswith(('.jpg')):
            FilePath = os.path.join(SmileFolder, f)
            data.append({"image_path": FilePath, "label": 0})

if os.path.exists(NonSmileFolder):
    for f in os.listdir(NonSmileFolder):
        if f.lower().endswith(('.jpg')):
            FilePath = os.path.join(NonSmileFolder, f)
            data.append({"image_path": FilePath, "label": 1})

df = pd.DataFrame(data)
df.to_csv("labeled_faces.csv", index=False)

print(f"Dataset successfully labeled! Total images processed: {len(df)}")
print("\n\nLabeled Smile Faces : ")
print(df.head())
print("\n\nLabeled Non-Smile Faces : ")
print(df.tail())

Dataset successfully labeled! Total images processed: 1202


Labeled Smile Faces : 
      image_path  label
0    smile\1.jpg      0
1   smile\10.jpg      0
2  smile\100.jpg      0
3  smile\101.jpg      0
4  smile\102.jpg      0


Labeled Non-Smile Faces : 
            image_path  label
1197  non_smile\95.jpg      1
1198  non_smile\96.jpg      1
1199  non_smile\97.jpg      1
1200  non_smile\98.jpg      1
1201  non_smile\99.jpg      1


<h3> Load Dataset

In [101]:
print("Loding Dataset...")
df = pd.read_csv("labeled_faces.csv")

Loding Dataset...


<h3> Helper Function - Useful to load actual image from path

In [102]:
def LoadProcess(image_paths):
    images = []
    
    for path in image_paths:
        try:
            img = load_img(path, target_size=(128,128))
            img_array = img_to_array(img) / 255.0 # Normalize Pixels Value
            images.append(img_array)
            
        except Exception as e:
            print(f"Skipping corrupt or missing file: {path}")
            
    return np.array(images)

<h3>Split Dataset (80:20)

In [103]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

X_train, y_train = LoadProcess(train_df['image_path'].values), train_df['label'].values
X_val, y_val = LoadProcess(val_df['image_path'].values), val_df['label'].values

<h3>Model Creation

In [104]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    
    layers.Dense(1, activation='sigmoid')
])

<h3> Model Compilation

In [105]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

<h3>Model Summary

In [106]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)                   │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_12 (MaxPooling2D)      │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_13 (Conv2D)                   │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_14 (MaxPooling2D)      │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_4 (Flatten)                  │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 128)                 │       3,211,392 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

<h3>Model Training

In [107]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val)
)

Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 19s 504ms/step - accuracy: 0.5827 - loss: 0.6773 - val_accuracy: 0.8174 - val_loss: 0.5305
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 20s 492ms/step - accuracy: 0.8127 - loss: 0.4605 - val_accuracy: 0.8672 - val_loss: 0.3027
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 20s 479ms/step - accuracy: 0.8918 - loss: 0.2894 - val_accuracy: 0.9004 - val_loss: 0.2099
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 23s 553ms/step - accuracy: 0.9022 - loss: 0.2697 - val_accuracy: 0.6100 - val_loss: 0.6979
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 15s 495ms/step - accuracy: 0.8564 - loss: 0.3649 - val_accuracy: 0.9419 - val_loss: 0.2125
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 20s 488ms/step - accuracy: 0.9313 - loss: 0.1931 - val_accuracy: 0.9585 - val_loss: 0.1758
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 21s 495ms/step - accuracy: 0.9417 - loss: 0.1681 - val_accuracy: 0.9253 - val_loss: 0.1631
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 21s 493ms/step - accuracy: 0.9272 - loss: 0.2222 - val_accu

<h3> Save Trained Parameters to Disk

In [108]:
model.save("FaceDetectorModel.h5")
print("Model saved as FaceDetectorModel.h5!")

Model saved as FaceDetectorModel.h5!


<h3>Load Model

In [109]:
model = load_model("FaceDetectorModel.h5")

<h3>Predict Model

In [110]:
def PredictFace(image_path):
    try:
        img = load_img(image_path, target_size=(128,128))
        img_array = img_to_array(img) / 255.0
        
        # Expand dimensions because Keras expects batches: (1, 128, 128, 3)
        img_array = np.expand_dims(img_array, axis=0)
        prediction = model.predict(img_array)[0][0]
        
        # 0 = Smile, 1 = Non-Smile
        if prediction < 0.5:
            confidence = (1 - prediction) * 100
            print(f"Image: {image_path}")
            print(f"Result: SMILING (Confidence: {confidence:.2f}%)\n")
        else:
            confidence = prediction * 100
            print(f"Image: {image_path}")
            print(f"Result: NOT SMILING (Confidence: {confidence:.2f}%)\n")
            
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")

<H3> Predict Faces from First 50 Test Images

In [111]:
image = [f"{i}.jpg" for i in range(1, 51)]
print(f"Loaded Images : {image}\n\n")
for name in image:
    image_path = os.path.join("test", name)
    PredictFace(image_path)

Loaded Images : ['1.jpg', '2.jpg', '3.jpg', '4.jpg', '5.jpg', '6.jpg', '7.jpg', '8.jpg', '9.jpg', '10.jpg', '11.jpg', '12.jpg', '13.jpg', '14.jpg', '15.jpg', '16.jpg', '17.jpg', '18.jpg', '19.jpg', '20.jpg', '21.jpg', '22.jpg', '23.jpg', '24.jpg', '25.jpg', '26.jpg', '27.jpg', '28.jpg', '29.jpg', '30.jpg', '31.jpg', '32.jpg', '33.jpg', '34.jpg', '35.jpg', '36.jpg', '37.jpg', '38.jpg', '39.jpg', '40.jpg', '41.jpg', '42.jpg', '43.jpg', '44.jpg', '45.jpg', '46.jpg', '47.jpg', '48.jpg', '49.jpg', '50.jpg']


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step
Image: test\1.jpg
Result: SMILING (Confidence: 99.62%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Image: test\2.jpg
Result: SMILING (Confidence: 100.00%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
Image: test\3.jpg
Result: SMILING (Confidence: 57.19%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
Image: test\4.jpg
Result: SMILING (Confidence: 78.02%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Image: test\5.jpg
Result: SMILING (Confidence: 57.53%)

1/1 ━━━━━━━━━━━━━━